# Market Basket Analysis (Association Rule Mining)

This notebook applies the Apriori algorithm to analyze patterns in genre selection, platform choices, and cross-domain connections. We explore which genres or platforms are frequently co-selected and identify interesting behaviors (e.g., users using specific platforms exclusively for specific genres).

## 1. Environment Setup & Imports

We load libraries for transactional processing (`mlxtend`), data handling (`pandas`), and dynamic workspace path resolution.

In [ ]:
import json
import sys
from pathlib import Path
import pandas as pd
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

## 2. Robust Path Loading & Ingestion

Instead of hardcoding path links, we use the project's standard path modules to locate the processed survey dataset.

In [ ]:
# Resolve repository root and register src path
notebook_dir = Path(__file__).resolve().parent if "__file__" in locals() else Path.cwd()
repo_root = notebook_dir.parent
if str(repo_root / "src") not in sys.path:
    sys.path.append(str(repo_root / "src"))

from kulture.common.paths import get_dataset_path

# Load cleaned survey data
data_path = get_dataset_path()
df = pd.read_csv(data_path)
print(f"Loaded {len(df)} rows from {data_path.name}")

## 3. Association Rule Mining Helper

We define a modular helper function, `get_rules`, which takes a pandas Series containing comma-separated multi-select items, encodes them into transaction representations, and applies Apriori to extract association rules based on minimum support and confidence thresholds.

In [ ]:
results = {}

def get_rules(series, min_support=0.1, min_threshold=0.5):
    # Drop NAs and "Unknown"
    series = series.dropna()
    series = series[~series.str.contains('Unknown', case=False)]
    
    # Split by comma and strip whitespace
    transactions = [[item.strip() for item in row.split(',')] for row in series]
    
    te = TransactionEncoder()
    te_ary = te.fit_transform(transactions)
    df_trans = pd.DataFrame(te_ary, columns=te.columns_)
    
    # Mine frequent itemsets
    frequent_itemsets = apriori(df_trans, min_support=min_support, use_colnames=True)
    if frequent_itemsets.empty:
        return []
        
    # Extract rules
    rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=min_threshold, num_itemsets=len(frequent_itemsets))
    
    output_rules = []
    for idx, row in rules.iterrows():
        antecedents = list(row['antecedents'])
        consequents = list(row['consequents'])
        support = float(row['support'])
        confidence = float(row['confidence'])
        lift = float(row['lift'])
        
        # Retain only rules with lift > 1.0 (positive correlation)
        if lift > 1.0:
            output_rules.append({
                'rule': f"IF {antecedents} THEN {consequents}",
                'support': support,
                'confidence': confidence,
                'lift': lift
            })
            
    # Sort by lift, then confidence
    output_rules.sort(key=lambda x: (x['lift'], x['confidence']), reverse=True)
    return output_rules

## 4. Analysis 1: Rules Within Genres

We investigate co-occurrence rules between listener music preferences.

In [ ]:
results['genre_rules'] = get_rules(df['top_genres'], min_support=0.1, min_threshold=0.6)
print(f"Found {len(results['genre_rules'])} genre rules.")
results['genre_rules'][:3]

## 5. Analysis 2: Rules Within Platforms

We find relationships between platforms used simultaneously by listeners.

In [ ]:
results['platform_rules'] = get_rules(df['primary_platforms'], min_support=0.1, min_threshold=0.6)
print(f"Found {len(results['platform_rules'])} platform rules.")
results['platform_rules'][:3]

## 6. Analysis 3: Cross-Domain Association Rules

We map intersections where selecting specific genres implies a preference for a platform, or vice versa.

In [ ]:
# Merge both fields into combined transaction sets
combined = df.dropna(subset=['top_genres', 'primary_platforms'])
combined = combined[~combined['top_genres'].str.contains('Unknown') & ~combined['primary_platforms'].str.contains('Unknown')]

cross_transactions = []
for _, row in combined.iterrows():
    genres = [g.strip() + " (Genre)" for g in row['top_genres'].split(',')]
    platforms = [p.strip() + " (Platform)" for p in row['primary_platforms'].split(',')]
    cross_transactions.append(genres + platforms)
    
te_cross = TransactionEncoder()
te_ary_cross = te_cross.fit_transform(cross_transactions)
df_cross = pd.DataFrame(te_ary_cross, columns=te_cross.columns_)

freq_cross = apriori(df_cross, min_support=0.15, use_colnames=True)
rules_cross = association_rules(freq_cross, metric="confidence", min_threshold=0.6, num_itemsets=len(freq_cross))

cross_output = []
for idx, row in rules_cross.iterrows():
    antecedents = list(row['antecedents'])
    consequents = list(row['consequents'])
    
    # Filter for rules crossing between the two domains
    ant_types = set([x.split('(')[-1] for x in antecedents])
    con_types = set([x.split('(')[-1] for x in consequents])
    
    if ant_types != con_types:
        support = float(row['support'])
        confidence = float(row['confidence'])
        lift = float(row['lift'])
        
        if lift > 1.1:
            cross_output.append({
                'rule': f"IF {antecedents} THEN {consequents}",
                'support': support,
                'confidence': confidence,
                'lift': lift
            })

cross_output.sort(key=lambda x: (x['lift'], x['confidence']), reverse=True)
results['genre_platform_rules'] = cross_output[:10]
print(f"Extracted {len(results['genre_platform_rules'])} cross-domain rules.")

## 7. Results Export

We serialize the mined rules to `market_basket_results.json` for consumption by the downstream reporting layers.

In [ ]:
with open('market_basket_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Market Basket Analysis rules successfully serialized.")